In [19]:
import pymupdf4llm
import re
import json
import os
from typing import List, Dict, Any, Tuple, Set
from pathlib import Path
from collections import defaultdict

# Load ontology keywords
def load_ontology_keywords(keywords_path: str) -> Dict[str, List[str]]:
    """Load ontology keywords from JSON file."""
    try:
        with open(keywords_path, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f"Warning: Could not load ontology keywords: {e}")
        return {"all_unique": []}

# Define spatial and offshore wind keywords for section filtering
SPATIAL_KEYWORDS = [
    "distance", "spacing", "separation", "buffer", "setback", "clearance", "proximity",
    "minimum distance", "maximum distance", "radius", "zone", "boundary", "perimeter",
    "interval", "gap", "apart", "between turbines", "turbine spacing", "layout",
    "array", "configuration", "positioning", "placement", "location", "coordinate",
    "meter", "metre", "kilometer", "kilometre", "mile", "nautical mile", "feet", "yard"
]

OFFSHORE_WIND_KEYWORDS = [
    "offshore", "wind farm", "wind turbine", "turbine", "wind energy", "renewable energy",
    "marine", "ocean", "sea", "coastal", "continental shelf", "water depth", "seabed",
    "foundation", "platform", "substation", "cable", "transmission", "grid connection",
    "lease area", "wind resource", "wind speed", "rotor", "blade", "nacelle", "tower"
]

REGULATION_KEYWORDS = [
    "regulation", "requirement", "standard", "guideline", "code", "rule", "law",
    "policy", "directive", "order", "shall", "must", "required", "mandatory",
    "compliance", "specification", "criterion", "provision", "condition",
    "permit", "license", "authorization", "approval", "restriction", "limitation"
]

def calculate_relevance_score(text: str, ontology_keywords: List[str]) -> float:
    """Calculate relevance score based on keyword presence and density."""
    text_lower = text.lower()
    words = re.findall(r'\b\w+\b', text_lower)
    word_count = len(words)
    
    if word_count == 0:
        return 0.0
    
    # Count keyword matches
    spatial_matches = sum(1 for kw in SPATIAL_KEYWORDS if kw in text_lower)
    offshore_matches = sum(1 for kw in OFFSHORE_WIND_KEYWORDS if kw in text_lower)
    regulation_matches = sum(1 for kw in REGULATION_KEYWORDS if kw in text_lower)
    ontology_matches = sum(1 for kw in ontology_keywords if kw.lower() in text_lower)
    
    # Weight different types of keywords
    score = (
        spatial_matches * 3.0 +      # Spatial keywords are most important
        offshore_matches * 2.0 +     # Offshore wind context
        regulation_matches * 1.5 +   # Regulatory language
        ontology_matches * 1.0       # Domain-specific terms
    ) / word_count * 100  # Normalize by text length
    
    return min(score, 100.0)  # Cap at 100

def has_spatial_indicators(text: str) -> bool:
    """Simple check if text likely contains spatial information - let LLM do the detailed extraction."""
    text_lower = text.lower()
    
    # Look for spatial context indicators
    spatial_indicators = [
        'distance', 'spacing', 'separation', 'meter', 'metre', 'km', 'feet', 'mile',
        'turbine', 'setback', 'buffer', 'clearance', 'zone', 'boundary', 'layout',
        'positioning', 'placement', 'array', 'configuration'
    ]
    
    # Check for numbers with units (basic pattern)
    has_measurements = bool(re.search(r'\d+(?:\.\d+)?\s*(?:m|meter|metre|km|kilometer|kilometre|ft|feet|mile|nautical\s*mile)s?\b', text_lower))
    
    # Check for spatial context words
    has_spatial_context = any(indicator in text_lower for indicator in spatial_indicators)
    
    return has_measurements or has_spatial_context

def extract_document_sections(text: str) -> List[Dict[str, Any]]:
    """Extract document sections with headers and content."""
    sections = []
    
    # Split by headers (markdown style)
    header_pattern = r'^(#{1,6})\s+(.+)$'
    lines = text.split('\n')
    
    current_section = {'level': 0, 'title': 'Document Start', 'content': [], 'start_pos': 0}
    current_pos = 0
    
    for i, line in enumerate(lines):
        line_start = current_pos
        current_pos += len(line) + 1  # +1 for newline
        
        header_match = re.match(header_pattern, line)
        
        if header_match:
            # Save previous section if it has content
            if current_section['content']:
                current_section['content'] = '\n'.join(current_section['content'])
                current_section['end_pos'] = line_start
                sections.append(current_section)
            
            # Start new section
            level = len(header_match.group(1))
            title = header_match.group(2).strip()
            current_section = {
                'level': level,
                'title': title,
                'content': [],
                'start_pos': line_start,
                'header_line': line
            }
        else:
            current_section['content'].append(line)
    
    # Add final section
    if current_section['content']:
        current_section['content'] = '\n'.join(current_section['content'])
        current_section['end_pos'] = current_pos
        sections.append(current_section)
    
    return sections

def extract_spatial_document(pdf_path: str, ontology_keywords_path: str, 
                           relevance_threshold: float = 2.0) -> Dict[str, Any]:
    """Extract spatial-related sections from document for LLM processing."""
    
    # Load ontology keywords
    ontology_data = load_ontology_keywords(ontology_keywords_path)
    ontology_keywords = ontology_data.get('all_unique', [])
    
    # Extract text from PDF
    try:
        md_text = pymupdf4llm.to_markdown(pdf_path)
    except Exception as e:
        print(f"Error processing PDF {pdf_path}: {e}")
        return {}
    
    doc_id = Path(pdf_path).stem
    print(f"  Document length: {len(md_text)} characters")
    
    # Extract document sections
    sections = extract_document_sections(md_text)
    print(f"  Found {len(sections)} sections")
    
    # Define sections to exclude (not useful for LLM spatial analysis)
    exclude_patterns = [
        'contents', 'table of contents', 'toc', 'list of tables', 'list of figures',
        'references', 'bibliography', 'appendix a', 'appendix b', 'appendix c',
        'acknowledgments', 'executive summary', 'abstract', 'acronyms', 'abbreviations'
    ]
    
    # Analyze sections for spatial relevance
    spatial_sections = []
    sections_with_spatial_indicators = 0
    
    for i, section in enumerate(sections):
        # Skip excluded sections
        section_title_lower = section['title'].lower()
        if any(pattern in section_title_lower for pattern in exclude_patterns):
            print(f"    Skipping excluded section: '{section['title']}'")
            continue
        
        # Skip very short sections (likely incomplete or headers only)
        if len(section['content'].strip()) < 100:
            print(f"    Skipping short section: '{section['title']}'")
            continue
        
        # Calculate relevance score
        score = calculate_relevance_score(section['content'], ontology_keywords)
        
        # Check for spatial indicators (simple check - let LLM do detailed extraction)
        has_spatial = has_spatial_indicators(section['content'])
        
        # Include section based on refined criteria
        include_section = False
        reason = ""
        
        if has_spatial:
            include_section = True
            reason = f"spatial_indicators_found"
            sections_with_spatial_indicators += 1
        elif score >= relevance_threshold * 3:  # Higher threshold for non-spatial content
            # Only include high-relevance non-spatial sections if they contain regulatory language
            regulation_indicators = ['shall', 'must', 'required', 'requirement', 'standard', 
                                   'specification', 'criterion', 'compliance', 'permit']
            if any(indicator in section['content'].lower() for indicator in regulation_indicators):
                include_section = True
                reason = f"high_relevance_regulatory"
        elif any(keyword in section['content'].lower() for keyword in 
                ['definition', 'definitions'] + SPATIAL_KEYWORDS[:5]):  # Key spatial definitions only
            if score >= relevance_threshold:
                include_section = True
                reason = f"spatial_definitions"
        
        if include_section:
            # Ensure section content is complete (ends with punctuation or clear break)
            content = section['content'].strip()
            
            # Check if content appears to be cut off mid-sentence
            if content and not content[-1] in '.!?':
                # Try to find a better ending point
                last_period = content.rfind('.')
                last_paragraph = content.rfind('\n\n')
                
                if last_period > len(content) * 0.8:  # Period is near the end
                    content = content[:last_period + 1]
                elif last_paragraph > len(content) * 0.7:  # Use last complete paragraph
                    content = content[:last_paragraph]
                # Otherwise keep as is - might be a list or table
            
            # Store section with minimal data - level only used temporarily for grouping
            section_data = {
                'title': section['title'],
                'content': content,
                'header_line': section.get('header_line', ''),
                '_temp_level': section['level']  # Temporary field for grouping
            }
            
            spatial_sections.append(section_data)
            
            print(f"    Section {i+1}: '{section['title'][:50]}...' - {reason}")
    
    print(f"  Sections with spatial indicators: {sections_with_spatial_indicators}")
    print(f"  Total sections included: {len(spatial_sections)}")
    
    # Create filtered document optimized for LLM processing
    filtered_content = []
    
    # Add document header with metadata
    filtered_content.append(f"# Spatial Requirements Analysis - {doc_id}")
    filtered_content.append(f"*Document filtered for spatial information relevant to offshore wind development*")
    filtered_content.append(f"*Sections included: {len(spatial_sections)} | Sections with spatial indicators: {sections_with_spatial_indicators}*")
    filtered_content.append(f"*Ready for LLM processing to extract detailed spatial requirements*\n")
    
    # Group sections by level for better organization
    main_sections = []
    subsections = []
    
    for section in spatial_sections:
        if section['_temp_level'] <= 3:  # Main sections (H1, H2, H3)
            main_sections.append(section)
        else:  # Subsections (H4, H5, H6)
            subsections.append(section)
    
    # Add main sections first
    for section in main_sections:
        # Remove temporary level field before adding to output
        clean_section = {k: v for k, v in section.items() if not k.startswith('_temp')}
        
        if section['header_line']:
            filtered_content.append(section['header_line'])
        else:
            # Reconstruct header from temp level
            header_prefix = '#' * max(1, section['_temp_level'])
            filtered_content.append(f"{header_prefix} {section['title']}")
        
        filtered_content.append(section['content'])
        filtered_content.append("")  # Empty line for separation
    
    # Add subsections
    if subsections:
        filtered_content.append("## Additional Relevant Subsections")
        filtered_content.append("")
        
        for section in subsections:
            if section['header_line']:
                filtered_content.append(section['header_line'])
            else:
                header_prefix = '#' * max(1, min(section['_temp_level'], 4))  # Limit depth
                filtered_content.append(f"{header_prefix} {section['title']}")
            
            filtered_content.append(section['content'])
            filtered_content.append("")
    
    filtered_document = '\n'.join(filtered_content)
    
    # Clean all sections before returning (remove temporary fields)
    clean_sections = []
    for section in spatial_sections:
        clean_section = {k: v for k, v in section.items() if not k.startswith('_temp')}
        clean_sections.append(clean_section)
    
    return {
        'doc_id': doc_id,
        'original_length': len(md_text),
        'filtered_length': len(filtered_document),
        'total_sections': len(sections),
        'spatial_sections': len(spatial_sections),
        'sections_with_spatial_indicators': sections_with_spatial_indicators,
        'filtered_document': filtered_document,
        'sections': clean_sections  # Clean sections without temporary fields
    }

def test_spatial_extraction(pdf_path: str, ontology_keywords_path: str, 
                          relevance_threshold: float = 2.0):
    """Test spatial section extraction on a single document."""
    
    print(f"=== TESTING SPATIAL SECTION EXTRACTION ===")
    print(f"File: {pdf_path}")
    
    result = extract_spatial_document(pdf_path, ontology_keywords_path, relevance_threshold)
    
    if result:
        print(f"\n=== EXTRACTION RESULTS ===")
        print(f"Document ID: {result['doc_id']}")
        print(f"Original length: {result['original_length']:,} characters")
        print(f"Filtered length: {result['filtered_length']:,} characters")
        print(f"Compression ratio: {result['filtered_length']/result['original_length']*100:.1f}%")
        print(f"Total sections: {result['total_sections']}")
        print(f"Spatial sections: {result['spatial_sections']}")
        print(f"Sections with spatial indicators: {result['sections_with_spatial_indicators']}")
        
        print(f"\n=== INCLUDED SECTIONS ===")
        for i, section in enumerate(result['sections'][:5]):  # Show first 5 sections
            print(f"Section {i+1}: {section['title']}")
            print(f"  Length: {len(section['content'])} chars")
            print(f"  Preview: {section['content'][:100]}...")
            print()
        
        if len(result['sections']) > 5:
            print(f"... and {len(result['sections']) - 5} more sections")
        
        print(f"\n=== FILTERED DOCUMENT PREVIEW ===")
        print(result['filtered_document'][:1000] + "...")
    else:
        print("No spatial content found!")
    
    return result

def process_all_documents(pdf_folder: str, output_folder: str, 
                         ontology_keywords_path: str, relevance_threshold: float = 2.0):
    """Process all PDFs in folder for LLM-ready spatial content."""
    
    pdf_folder = Path(pdf_folder)
    output_folder = Path(output_folder)
    output_folder.mkdir(exist_ok=True)
    
    # Find all PDF files
    pdf_files = list(pdf_folder.glob("*.pdf"))
    
    if not pdf_files:
        print(f"No PDF files found in {pdf_folder}")
        return
    
    print(f"Processing {len(pdf_files)} PDF files for LLM-ready spatial content...")
    
    total_docs_processed = 0
    docs_with_spatial_content = 0
    
    # Process each PDF
    for pdf_file in pdf_files:
        try:
            print(f"\nProcessing: {pdf_file.name}")
            
            result = extract_spatial_document(
                str(pdf_file), 
                ontology_keywords_path, 
                relevance_threshold
            )
            
            if result and result['spatial_sections'] > 0:
                # Only save metadata - the filtered_document content is processed directly by LLM
                metadata_file = output_folder / f"{result['doc_id']}_metadata.json"
                with open(metadata_file, 'w', encoding='utf-8') as f:
                    json.dump(result, f, indent=2, ensure_ascii=False)
                
                print(f"  ✓ Processed document: {result['doc_id']}")
                print(f"  ✓ Sections: {result['spatial_sections']} | Spatial indicators: {result['sections_with_spatial_indicators']}")
                print(f"  ✓ Metadata saved: {metadata_file.name}")
                
                docs_with_spatial_content += 1
            else:
                print(f"  ⚠ No relevant spatial content found")
            
            total_docs_processed += 1
                
        except Exception as e:
            print(f"  ✗ Error processing {pdf_file.name}: {e}")
    
    print(f"\n=== PROCESSING SUMMARY ===")
    print(f"Documents processed: {total_docs_processed}/{len(pdf_files)}")
    print(f"Documents with spatial content: {docs_with_spatial_content}")
    print(f"Ready for LLM processing: {docs_with_spatial_content} documents")

# Configuration and execution
PDF_FOLDER = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Documents"
OUTPUT_FOLDER = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Processed_Results_V1"
ONTOLOGY_PATH = "/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Extract_regulations/Filtered_regulations/ontology_keywords.json"
RELEVANCE_THRESHOLD = 2.0

# Test spatial extraction on a single document
TEST_MODE = True  # Set to False to process all documents

if __name__ == "__main__":
    if TEST_MODE:
        # Test with the first PDF file found
        pdf_folder = Path(PDF_FOLDER)
        pdf_files = list(pdf_folder.glob("*.pdf"))
        
        if pdf_files:
            test_file = pdf_files[0]  # Take the first PDF
            print(f"Testing spatial extraction with: {test_file.name}")
            
            result = test_spatial_extraction(
                str(test_file), 
                ONTOLOGY_PATH, 
                RELEVANCE_THRESHOLD
            )
            
            if result:
                # Save test results - only metadata needed
                output_folder = Path(OUTPUT_FOLDER)
                output_folder.mkdir(exist_ok=True)
                
                # Save complete result including filtered_document content
                metadata_file = output_folder / f"test_{result['doc_id']}_complete.json"
                with open(metadata_file, 'w', encoding='utf-8') as f:
                    json.dump(result, f, indent=2, ensure_ascii=False)
                
                print(f"\nTest document ready for LLM processing")
                print(f"Complete data saved to: {metadata_file}")
        else:
            print("No PDF files found for testing!")
    
    else:
        # Process all PDFs for LLM-ready spatial content
        process_all_documents(
            PDF_FOLDER, 
            OUTPUT_FOLDER, 
            ONTOLOGY_PATH, 
            RELEVANCE_THRESHOLD
        )
        
        print("\n=== PROCESSING COMPLETE ===")
        print(f"Check {OUTPUT_FOLDER} for LLM-ready documents")

Testing spatial extraction with: 49.pdf
=== TESTING SPATIAL SECTION EXTRACTION ===
File: /Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/Documents/49.pdf
  Document length: 635754 characters
  Found 108 sections
    Skipping short section: 'December 2022 U.S. Department of the Interior  Bureau of Ocean Energy Management  Office of Renewable Energy Program'
    Skipping excluded section: 'Contents'
    Section 3: 'Figures...' - spatial_indicators_found
    Skipping excluded section: 'Appendix A Tables'
    Skipping excluded section: 'Acronyms and Abbreviations'
    Section 8: '**2.0 Proposed Action **...' - spatial_indicators_found
    Section 10: '**2.2 Construction and Installation **...' - spatial_indicators_found
    Section 11: '2.2.1 Installation of WTG/OSS Structures and Found...' - spatial_indicators_found
    Section 12: '2.2.1.1 Vessel Activity...' - spatial_indicators_found
    Section 13: '2.2.1.2 Pile Driving...' - spatial_indicators_fou